In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [2]:
FINESS = pd.read_excel("finess.xlsx")
FINESS = FINESS.drop(FINESS.columns[[1,2,4]],axis=1)
FINESS = FINESS.rename(columns={"FINESS":"FI","Statut Juridique":"Statut"})

Urg2022 = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
Urg2022 = Urg2022.merge(FINESS, on='FI', how='left')

Data = Urg2022[Urg2022['URG']=='GEN']  # On ne garde que les urgences générales
Data = Data[['FI','PASSU','HMED','HIDE','Statut']]

In [3]:
Data.describe()

,PASSU,HMED,HIDE
count,607.000000,601.000000,602.000000
mean,31518.583196,373.896839,745.141196
std,19895.436515,263.481934,855.306672
min,2069.000000,10.000000,14.000000
25%,16488.500000,168.000000,336.000000
50%,26224.000000,310.000000,565.000000
75%,41926.500000,476.000000,938.500000
max,113596.000000,3360.000000,12775.000000


On prépare les variables pour la régression :

In [4]:
Data["lPASSU"] = np.where(Data["PASSU"] > 0, np.log(Data["PASSU"]), 0)
Data["lHIDE"]  = np.where(Data["HIDE"]  > 0, np.log(Data["HIDE"]),  0)
Data["lHMED"]  = np.where(Data["HMED"]  > 0, np.log(Data["HMED"]),  0)

On construit deux variables de contrôle : Statut_PNL (resp. Statut_PL) vaut 1 si l'établissement est privé non lucratif (resp. privé lucratif) 

In [5]:
dummies = pd.get_dummies(Data["Statut"], prefix="Statut")

Data["Statut_PNL"] = dummies["Statut_Privé non lucratif"]
Data["Statut_PL"]  = dummies["Statut_Privé lucratif"]

Data["Statut_PNL"] = Data["Statut_PNL"].astype(int)
Data["Statut_PL"]  = Data["Statut_PL"].astype(int)

Data["lHIDE_PNL"] = Data["lHIDE"] * Data["Statut_PNL"]
Data["lHIDE_PL"]  = Data["lHIDE"] * Data["Statut_PL"]
Data["lHMED_PNL"] = Data["lHMED"] * Data["Statut_PNL"]
Data["lHMED_PL"]  = Data["lHMED"] * Data["Statut_PL"]

In [6]:
Data

,FI,PASSU,HMED,HIDE,Statut,lPASSU,lHIDE,lHMED,Statut_PNL,Statut_PL,lHIDE_PNL,lHIDE_PL,lHMED_PNL,lHMED_PL
0,010000024,42384,930.0,733.0,Public,10.654526,6.597146,6.835185,0,0,0.0,0.000000,0.0,0.000000
1,010000032,16727,168.0,168.0,Public,9.724779,5.123964,5.123964,0,0,0.0,0.000000,0.0,0.000000
2,010005239,22583,420.0,504.0,Public,10.024953,6.222576,6.040255,0,0,0.0,0.000000,0.0,0.000000
3,010780195,23369,190.0,336.0,Privé lucratif,10.059166,5.817111,5.247024,0,1,0.0,5.817111,0.0,5.247024
4,010780203,38727,369.0,898.0,Privé lucratif,10.564292,6.800170,5.910797,0,1,0.0,6.800170,0.0,5.910797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,970400024,36008,446.0,975.0,Public,10.491496,6.882437,6.100319,0,0,0.0,0.000000,0.0,0.000000
687,970400057,48574,485.0,1176.0,Public,10.790844,7.069874,6.184149,0,0,0.0,0.000000,0.0,0.000000
689,970400065,35983,450.0,903.0,Public,10.490802,6.805723,6.109248,0,0,0.0,0.000000,0.0,0.000000
691,970400073,31213,168.0,33.0,Public,10.348590,3.496508,5.123964,0,0,0.0,0.000000,0.0,0.000000


On réalise une regression linéaire MCO avec les variables de contrôle :

In [7]:
X = Data[["lHIDE", "lHMED","Statut_PNL", "Statut_PL","lHIDE_PNL", "lHIDE_PL","lHMED_PNL", "lHMED_PL"]]
X = sm.add_constant(X)

y = Data["lPASSU"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 lPASSU   R-squared:                       0.423
Model:                            OLS   Adj. R-squared:                  0.415
Method:                 Least Squares   F-statistic:                     54.78
Date:                Fri, 30 Jan 2026   Prob (F-statistic):           1.42e-66
Time:                        10:50:13   Log-Likelihood:                -436.88
No. Observations:                 607   AIC:                             891.8
Df Residuals:                     598   BIC:                             931.4
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.5959      0.147     51.677      0.0

Pour tester si le statut influence l'efficacité des établissements :

In [8]:
model.f_test("Statut_PNL = 0, Statut_PL = 0")

<class 'statsmodels.stats.contrast.ContrastResults'>
<F test: F=5.05635085478554, p=0.006643786832430214, df_denom=598, df_num=2>

Calcul des différences d'efficacité :

In [9]:
coefs = model.params
gamma_pnl = coefs["Statut_PNL"]
gamma_pl  = coefs["Statut_PL"]

eff_pnl = (np.exp(gamma_pnl) - 1) * 100
eff_pl  = (np.exp(gamma_pl)  - 1) * 100

print(f"Privé non lucratif vs Public : {eff_pnl:.2f} %")
print(f"Privé lucratif vs Public     : {eff_pl:.2f} %")

Privé non lucratif vs Public : 13.39 %
Privé lucratif vs Public     : -88.30 %


Valeur très étonnante pour le privé lucratif, cela provient il des données ? Code à tester sur les données 2023 pour confirmer.

Cette valeur s'explique peut être par le fait que les hopitaux privés emploi du personnel libéral et ne comptent pas les heures de la même façon. On va réaliser une autre régression en utilisant les valeurs de la table 'URGENCES_P' et compter les effectifs en personnel libéral et non libéral.

In [10]:
UrgP2022 = pd.read_csv("SAE/2022/URGENCES_P_2022a.csv", sep=";",encoding="latin-1")

In [14]:
UrgP2022[UrgP2022['PERSO'].isin(["M1000","M2000","M3020"])].count()

BOR        51
AN         51
FI         51
RS         51
FI_EJ      51
PERSO      51
ETPSAL     50
EFFLIB     12
MOD         0
COMSMUR     0
dtype: int64

comptabilsation des Médecins (catégories de personnel M1000, M2000 et M3020) par établissement en 2022, pour les salariés :

In [16]:
Medecins2022 = UrgP2022[UrgP2022["PERSO"].isin(["M1000","M2000","M3020"])]
ETP_Medecins2022 = Medecins2022.groupby("FI",as_index=False)["ETPSAL"].sum()
ETP_Medecins2022

,FI,ETPSAL
0,020000162,1.00
1,060000510,1.00
2,060000544,0.60
3,060780947,0.10
4,060785003,4.20
5,140000209,1.00
6,170000087,3.00
7,210987558,1.00
8,240000489,1.00
9,250006954,2.27


In [17]:
EFFLIB_Medecins2022 = Medecins2022.groupby("FI",as_index=False)["EFFLIB"].sum()
EFFLIB_Medecins2022

,FI,EFFLIB
0,020000162,0.0
1,060000510,0.0
2,060000544,0.0
3,060780947,0.0
4,060785003,0.0
5,140000209,0.0
6,170000087,0.0
7,210987558,0.0
8,240000489,0.0
9,250006954,0.0
